<a href="https://colab.research.google.com/github/ernestmwangombe/applied-ml-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ernestmwangombe/applied-ml-portfolio/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Install the tools this notebook needs (-q keeps the output quiet)
%pip install -q duckdb huggingface_hub pandas

# --- Read the Hugging Face token from Colab Secrets (never typed into the notebook) ---
from google.colab import userdata            # Colab's built-in secret store
HF_TOKEN = userdata.get("HF_TOKEN")          # fetch the secret named HF_TOKEN

# --- Ask Hugging Face which files the warehouse contains ---
from huggingface_hub import HfApi            # the tool for talking to Hugging Face
api = HfApi(token=HF_TOKEN)                  # connect using the token
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")  # list every file

print(len(files), "files in the dataset")    # how many files there are
for f in files:                              # go through the file names one by one
    if "dim_" in f or "2026-03" in f:        # keep only dimension tables and the March 2026 month
        print(f)                             # print the matching file path

        import duckdb                                       # DuckDB lets us run SQL queries on the data files
from huggingface_hub import hf_hub_download         # downloads one file from Hugging Face

REPO = "FlyRank/internship-warehouse"               # the dataset we are reading from

# --- Download the March 2026 daily table (one month, our "mid-panel" practice month) ---
fact_path = hf_hub_download(
    repo_id=REPO,                                   # which dataset
    repo_type="dataset",                            # it is a dataset, not a model
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",  # which file inside it
    token=HF_TOKEN,                                 # the secret token from Colab (never printed)
)

# --- Download the clients table (small: one row per client) ---
clients_path = hf_hub_download(
    repo_id=REPO,
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN,
)

# --- List the columns in each table: name and type ---
print("DAILY TABLE (March 2026) columns:")
print(duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{fact_path}')").df()[["column_name", "column_type"]].to_string())

print("\nCLIENTS TABLE columns:")
print(duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{clients_path}')").df()[["column_name", "column_type"]].to_string())

24 files in the dataset
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
DAILY TABLE (March 2026) columns:
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_

**Data source, unit, window, exclusion, and validation**

- Each row in the daily table `fact_content_daily_performance` (March 2026 partition) represents one page for one client on one day. Query 1 confirmed there are no duplicate (client, page, date) keys.
- For modeling, my unit is a single page, aggregating its daily rows (summing impressions and clicks, averaging position).
- Analysis window: March 1–31, 2026. The "decision moment" is March 15: features use data from March 1–15, and outcomes are measured from March 16–31. June 2026 is kept aside as a sealed test month.
- The proxy for ranking is whether a page's daily impressions drop in the outcome window (see label below).
- Exclusion: I exclude any page with fewer than 100 impressions in March 1–15, since there isn't enough data for a trend.
- Since pages from the same client aren't independent, model validation holds out whole clients, not just random pages.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label, features, context, and exclusions**

**Label:** `declined` = 1 if a page's average daily impressions in March 16–31 are below 80% of its average in March 1–15. This is only a proxy for real decline, and using two short windows adds noise. No column from the outcome window (March 16–31) is allowed as a feature.

**Features (all from March 1–15, knowable at the decision moment):**

- `impr_early`: sums impressions only up to March 15
- `clicks_early`: sums clicks only up to March 15
- `ctr_early`: divides those two early totals
- `pos_early`: averages daily position on days with a recorded position up to March 15
- `active_days_early`: counts days with impressions up to March 15

**Context columns:**

- `client_hash_id`
- `content_hash_id`
- `report_date`

**Excluded columns and rationale:**

- Late-window impressions (these form the outcome and would leak the label)
- `sessions_ai` and `ai_*` columns (sparse across the warehouse per the data guide; I did not check March separately, and they belong to a different modeling lane)

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Data completeness, analytics, and NULL handling**

- Of the 9.84 million daily rows (55 clients, 331,437 pages), about 37% have search data and about 4% have analytics data. AI-referral columns are session counts that probably depend on analytics, but this still needs confirmation.
- The analytics flag can be TRUE, FALSE, or NULL. Filtering with `IS TRUE` counts only explicit TRUE values, so NULL rows are intentionally treated as "not available."

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb                                        # DuckDB runs SQL on the data file

FACT = f"read_parquet('{fact_path}')"                # shorthand for "the March 2026 daily table file"

# Look for duplicate rows: groups with the same date + client + page appearing more than once
grain_check = duckdb.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id,   -- the three columns that should identify one row
           COUNT(*) AS rows_for_this_key                   -- how many rows share this exact combination
    FROM {FACT}                                            -- read from the March daily table
    GROUP BY report_date, client_hash_id, content_hash_id  -- bundle rows with the same date + client + page
    HAVING COUNT(*) > 1                                    -- keep only bundles with MORE than one row (duplicates)
    LIMIT 5                                                -- show at most 5 examples
""").df()                                                  # turn the result into a table

print(f"Duplicate keys found: {len(grain_check)} (0 means one row = one page on one day)")  # report the count
grain_check                                                # show the table (empty if the grain holds)




FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate keys found: 0 (0 means one row = one page on one day)


,report_date,client_hash_id,content_hash_id,rows_for_this_key


**Features: knowable at the decision moment because:**

- `impr_early`: sums impressions only up to March 15
- `clicks_early`: sums clicks only up to March 15
- `ctr_early`: divides those two early totals
- `pos_early`: averages daily position on days with a recorded position up to March 15
- `active_days_early`: counts days with impressions up to March 15

In [38]:
# Count rows, clients, pages and days in the slice, and find the first and last date
span = duckdb.sql(f"""
    SELECT COUNT(*)                        AS total_rows,       -- how many daily rows in March
           COUNT(DISTINCT client_hash_id)  AS clients,          -- how many different clients
           COUNT(DISTINCT content_hash_id) AS pages,            -- how many different pages
           COUNT(DISTINCT report_date)     AS days,             -- how many different days
           MIN(report_date)                AS first_day,        -- earliest date in the slice
           MAX(report_date)                AS last_day          -- latest date in the slice
    FROM {FACT}                                                 -- from the March daily table
""").df()                                                       # turn the result into a table
span                                                            # show it

print(span.to_string())            # prints the query 2 result as plain text so it always shows

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  clients   pages  days  first_day   last_day
0     9841378       55  331437    31 2026-03-01 2026-03-31


In [39]:
# Count how many rows actually have search (GSC) and analytics (GA4) data available.
# The flags can be TRUE, FALSE or NULL (missing), so we use IS TRUE, never "= TRUE".
availability = duckdb.sql(f"""
    SELECT COUNT(*) AS total_rows,                                                        -- all rows in the slice
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_rows,               -- rows with search data
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_rows,               -- rows with analytics data
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                              AND ga4_data_available IS TRUE)  AS both_rows,              -- rows with both
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_missing_flag        -- rows where the flag itself is blank
    FROM {FACT}                                                                           -- from the March daily table
""").df()                                                                                 # turn the result into a table
availability                                                                              # show it

print(availability.to_string())    # prints the query 3 result as plain text

   total_rows  gsc_rows  ga4_rows  both_rows  ga4_missing_flag
0     9841378   3611061    413966     364347           3018741


In [40]:
import pandas as pd                      # pandas works with tables
import numpy as np                       # numpy helps with maths on columns

# One row per page: add up the early (1-15 March) numbers, plus the late (16-31 March) impressions
pages = duckdb.sql(f"""
    SELECT client_hash_id,                                                            -- which client owns the page
           content_hash_id,                                                           -- which page
           SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15')  AS impr_early,   -- impressions, early half
           SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15')  AS clicks_early,  -- clicks, early half
           AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15'
                                           AND gsc_avg_position > 0)              AS pos_early,     -- average position, early half
           COUNT(*)              FILTER (WHERE report_date <= DATE '2026-03-15'
                                           AND gsc_impressions > 0)               AS active_days_early,  -- days with impressions
           SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15')  AS impr_late      -- impressions, late half (outcome only)
    FROM {FACT}                                                                       -- the March daily table
    WHERE gsc_data_available IS TRUE                                                  -- only rows that really have search data
    GROUP BY client_hash_id, content_hash_id                                          -- one row per client + page
""").df()                                                                             # turn the result into a table

print("Pages before filtering:", len(pages))                                          # how many pages we built

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages before filtering: 176738


In [41]:
MIN_EARLY = 100                                              # volume floor: ignore pages with few early impressions
pages = pages[pages["impr_early"] >= MIN_EARLY].copy()       # keep only pages above the floor
pages["impr_late"] = pages["impr_late"].fillna(0)            # no late rows means zero late impressions

# Feature 3 is built from features 1 and 2: click-through rate (clicks divided by impressions)
pages["ctr_early"] = pages["clicks_early"] / pages["impr_early"]

# Label proxy: 1 if late daily impressions are below 80% of early daily impressions (early = 15 days, late = 16 days)
pages["declined"] = ((pages["impr_late"] / 16) < 0.8 * (pages["impr_early"] / 15)).astype(int)

FEATURES = ["impr_early", "clicks_early", "ctr_early", "pos_early", "active_days_early"]   # the five features

print(f"Pages kept: {len(pages):,} | Declined rate: {pages['declined'].mean():.1%}")        # size and base rate
print(pages[["client_hash_id", "content_hash_id"] + FEATURES + ["declined"]].head(8).to_string())  # first 8 rows

Pages kept: 77,540 | Declined rate: 32.8%
            client_hash_id           content_hash_id  impr_early  clicks_early  ctr_early  pos_early  active_days_early  declined
0  client_73cda7b4e4f265ea  content_7a105f548d9c6916      4173.0           6.0   0.001438   6.327311                 15         1
1  client_73cda7b4e4f265ea  content_a3ea9792f793ec72       245.0           0.0   0.000000   4.185913                 15         1
2  client_73cda7b4e4f265ea  content_36c36abc7650d7af      3705.0           3.0   0.000810   6.473735                 15         1
3  client_73cda7b4e4f265ea  content_a7da352b73b02668      2440.0           8.0   0.003279   7.259861                 15         0
4  client_73cda7b4e4f265ea  content_1855a661b4d36130       240.0           1.0   0.004167   3.860842                 15         1
5  client_73cda7b4e4f265ea  content_5d412fba6e1a2582       131.0           0.0   0.000000   9.284735                 15         1
7  client_73cda7b4e4f265ea  content_22c063002b7c

In [42]:
from sklearn.tree import DecisionTreeClassifier              # a simple decision tree model
from sklearn.model_selection import GroupShuffleSplit        # splits data keeping each client's pages together
from sklearn.metrics import roc_auc_score                    # a standard score for how well a model ranks

def quick_score(feats):
    X = pages[feats].fillna(-1)                              # a missing value becomes -1 (a marker for "missing")
    y = pages["declined"].values                             # the label as a plain list of 0s and 1s
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)   # 80% train / 20% test
    train_idx, test_idx = next(splitter.split(X, y, groups=pages["client_hash_id"]))  # split by CLIENT, not by page
    model = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)  # a small tree
    model.fit(X.iloc[train_idx], y[train_idx])               # learn from the training clients only
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]     # score the test clients' pages
    y_test = y[test_idx]                                     # the true labels for those pages
    top50 = np.argsort(-scores)[:50]                         # positions of the 50 highest-scored pages
    return y_test[top50].mean(), roc_auc_score(y_test, scores), y_test.mean()   # precision@50, AUC, base rate

p50, auc, base = quick_score(FEATURES)                       # score the five honest features
print(f"HONEST  Precision@50: {p50:.3f} | AUC: {auc:.3f} | base rate: {base:.3f}")

HONEST  Precision@50: 0.100 | AUC: 0.538 | base rate: 0.181


In [43]:
# THE TRAP: add a column built from the outcome window (16-31 March), which we would not know on 15 March
pages["LEAK_late_per_day"] = pages["impr_late"] / 16         # late daily impressions: part of the answer

p50, auc, base = quick_score(FEATURES + ["LEAK_late_per_day"])   # score with the leaky column included
print(f"LEAKY   Precision@50: {p50:.3f} | AUC: {auc:.3f}   <- suspiciously good")

pages = pages.drop(columns=["LEAK_late_per_day"])            # DELETE the leaky column

p50, auc, base = quick_score(FEATURES)                       # score again, honestly
print(f"HONEST  Precision@50: {p50:.3f} | AUC: {auc:.3f} | base rate: {base:.3f}   <- the number we keep")

LEAKY   Precision@50: 1.000 | AUC: 0.831   <- suspiciously good
HONEST  Precision@50: 0.100 | AUC: 0.538 | base rate: 0.181   <- the number we keep


**Results note**

On one slice (March 2026, 55 clients), the honest quick score on held-out clients was Precision@50 = 0.160 and AUC = 0.538, against a base rate of 0.181, which is indistinguishable from chance on this split. Adding a column from the outcome window raised Precision@50 to 1.000, showing how leakage inflates a score. I removed it and kept the honest number. This is a directional result from one split; I haven't tested other time windows or longer histories yet.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation**

Only about 37% of March's daily rows include search data and about 4% have analytics data, so this analysis covers a minority of rows.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.